# Sales Analytics Dashboard - Data Validation (Python)

**Phase:** Python-based data validation (before SQL)

**Goal:** Verify dataset integrity and relational assumptions before modeling

This notebook validates:
- File availability and loadability
- Primary key uniqueness
- Table grain expectations
- Foreign key coverage (referential integrity)

> Single source of truth: executed cells + repository files

In [ ]:
# Standard library imports
# ------------------------------------
# os: interact with the operating system (working directory, file inspection)
# pathlib.Path: correct path handling
import os
from pathlib import Path

#Third-party library for data manipulation and analysis
import pandas as pd

# Display configuration
# ------------------------------------
# Increase column display limit for better inspection
pd.set_option("display.max_columns", 50)

# Adjust display width to avoid truncated tables in notebook view
pd.set_option("display.width", 120)

## Execution context

Notebooks depend on the kernel working directory.

This project avoids hard-coded absolute paths (e.g., '/Users/...') and resolves the repository root dynamically to keep the notebook portable.

In [ ]:
# Resolve project root dynamically (portable across machines and launch directories)
cwd = Path.cwd().resolve()

def find_project_root(start: Path) -> Path:
    for p in [start] + list(start.parents):
        if (p / "01-data").exists() and (p / "02-notebooks").exists():
            return p
    raise FileNotFoundError("Project root not found. Expected folders: '01-data' and '02-notebooks'.")
    
PROJECT_ROOT = find_project_root(cwd)
RAW_DATA_PATH = PROJECT_ROOT / "01-data" / "01-raw"

assert RAW_DATA_PATH.exists(), f"Raw data path does not exist: {RAW_DATA_PATH}"

PROJECT_ROOT, RAW_DATA_PATH

## Raw data availability

Before loading, we verify the expected CSV files exist in the raw data directory.

Fail fast > debug later.

In [ ]:
expected_files = {
    "orders": "orders_dataset.csv",
    "customers": "customers_dataset.csv",
    "order_items": "order_items_dataset.csv",
    "order_payments": "order_payments_dataset.csv",
    "order_reviews": "order_reviews_dataset.csv",
    "products": "product_summarize_dataset.csv",
}

existing = {p.name for p in RAW_DATA_PATH.glob("*.csv")}
missing = sorted([f for f in expected_files.values() if f not in existing])

assert not missing, f"Missing expected CSV files: {missing}"

sorted(existing)

## Data loading

At this stage we only load and validate structure.
No SQL modeling or BI work is performed until integrity checks pass.

In [ ]:
orders = pd.read_csv(RAW_DATA_PATH/expected_files["orders"])
customers = pd.read_csv(RAW_DATA_PATH/expected_files["customers"])
order_items = pd.read_csv(RAW_DATA_PATH/expected_files["order_items"])
order_payments = pd.read_csv(RAW_DATA_PATH/expected_files["order_payments"])
order_reviews = pd.read_csv(RAW_DATA_PATH/expected_files["order_reviews"])
products = pd.read_csv(RAW_DATA_PATH/expected_files["products"])

In [ ]:
datasets = {
    "orders": orders,
    "customers": customers,
    "order_items": order_items,
    "order_payments": order_payments,
    "order_reviews": order_reviews,
    "products": products
}

inventory = pd.DataFrame(
    [{"dataset": name, "rows": df.shape[0], "cols": df.shape[1]} for name, df in datasets.items()]
).sort_values("dataset").reset_index(drop=True)

inventory

In [ ]:
orders.head()

## Data validation - Structural integrity

We validate the analytical assumptions of the dataset:
1. Primary key uniqueness
2. Grain consistency across tables
3. Foreign key coverage (referential integrity)

All checks are logged as PASS/FAIL with supporting counts.

In [ ]:
results = []

def log_result(test_name: str, subject: str, passed: bool, details: str = "") -> None:
    results.append({
        "test_name": test_name,
        "subject": subject,
        "status": "PASS" if passed else "FAIL",
        "details": details,
    })

def results_df() -> pd.DataFrame:
    return pd.DataFrame(results)

In [ ]:
pk_unique = orders["order_id"].is_unique
dup_count = int(orders["order_id"].duplicated().sum())

log_result(
    test_name="pk_uniqueness",
    subject="orders.order_id",
    passed=pk_unique,
    details=f"duplicates={dup_count}, rows={len(orders)}"
)

pk_unique

In [ ]:
orders_total = len(orders)
orders_in_items = int(order_items["order_id"].nunique())

passed = orders_in_items <= orders_total

log_result(
    test_name="grain_check",
    subject="order_items.order_id vs orders.order_id",
    passed=passed,
    details=f"distinct_orders_in_items={orders_in_items}, orders_total={orders_total}"
)

orders_total, orders_in_items

## Foreign Key coverage (referential integrity)

These checks verify that every key value in each child table exists in the referenced parent table.

This notebook treats FK coverage as a structural constraint (tolerance = 0) unless explicity justified and documented.

**Note on customers:** 'customer_id' is used for techincal joins (orders -> customers).
'customer_unique_id' is reserved for business-level "unique customer" definitions in later analysis.

In [ ]:
def fk_coverage(child_df: pd.DataFrame, child_col: str, parent_df: pd.DataFrame, parent_col: str):
    missing_mask = ~child_df[child_col].isin(parent_df[parent_col])
    missing_count = int(missing_mask.sum())
    missing_rate = float(missing_count / len(child_df)) if len(child_df) else 0.0
    return missing_count, missing_rate

FK_TESTS = [
    ("orders", "customer_id", "customers", "customer_id"),
    ("order_items", "order_id", "orders", "order_id"),
    ("order_items", "product_id", "products", "product_id"),
    ("order_payments", "order_id", "orders", "order_id"),
    ("order_reviews", "order_id", "orders", "order_id")
]

for child_t, child_c, parent_t, parent_c in FK_TESTS:
    child_df = datasets[child_t]
    parent_df = datasets[parent_t]
    
    missing_count, missing_rate = fk_coverage(child_df, child_c, parent_df, parent_c)
    
    log_result(
        test_name="fk_coverage",
        subject=f"{child_t}.{child_c} -> {parent_t}.{parent_c}",
        passed=(missing_count == 0),
        details=f"child_rows={len(child_df)}, parent_rows={len(parent_df)}, missing_count={missing_count}, missing_rate={missing_rate:.4%}"
    )

results_df()


In [ ]:
results_df().sort_values(["status", "test_name", "subject"]).reset_index(drop=True)

## Interpretation of FK failures

Two FK checks failed:

- 'order_items.product_id -> products.product_id': The current products file appears to be a subset ('product_summarize_dataset.csv'), so it does not cover the full product universe referenced by order items. This blocks a complete product dimension unless a full extract is provided.

- 'order_reviews.order_id -> orders.order_id': Some review records reference orders not present in the current orders dataset. We quantify the gap and decide whether to (a) treat reviews as out-of-scope for the current model, or (b) apply a documented filter to align datasets.

Next step: quantify missing counts/rates and document the chosen modeling rule.

In [ ]:
missing_products = set(order_items["product_id"]) - set(products["product_id"])
len(missing_products)

In [ ]:
missing_count = (~order_items["product_id"].isin(products["product_id"])).sum()
missing_rate = missing_count / len(order_items)
missing_count, f"{missing_rate:.2%}"

In [ ]:
missing_review_orders = set(order_reviews["order_id"]) - set(orders["order_id"])
len(missing_review_orders)

In [ ]:
missing_count = (~order_reviews["order_id"].isin(orders["order_id"])).sum()
missing_rate = missing_count / len(order_reviews)
missing_count, f"{missing_rate:.2%}"

## FK Gap Assesment and Modeling Decision

Foreign key validation revealed:

- 1.42% of 'order_items' reference product_ids not present in the current products dataset.
- 2 review records reference orders not present in 'orders'.

For analytical modeling purposes, referential integrity will be enforced strictly. Child tables will be filtered to match available parent keys.

This ensures a consistent star sc hema while making data loss explicit and measurable.

In [ ]:
order_items_model = order_items[
    order_items["product_id"].isin(products["product_id"])
].copy()

order_reviews_model = order_reviews[
    order_reviews["order_id"].isin(orders["order_id"])
].copy()

len(order_items), len(order_items_model), len(order_reviews), len(order_reviews_model)

## Primary Key validation (remaining tables)

We validate uniqueness for:
- 'customers.customer_id'
- 'products.product_id'
- 'order_reviews.review_id'

For 'order_items', the expected unique identifier is the composite key:
- ('order_id', 'order_item_id')

In [ ]:
# PK uniqueness checks
customers_pk_unique = customers["customer_id"].is_unique
products_pk_unique = products["product_id"].is_unique
reviews_pk_unique = order_reviews["review_id"].is_unique

# Composite PK uniqueness check for order_items
order_items_composite_unique = ~order_items.duplicated(subset=["order_id","order_item_id"]).any()

customers_pk_unique, products_pk_unique, reviews_pk_unique, order_items_composite_unique

## Relationship multiplicites (informational)

These counts help confirm expected relationship patterns:
- Items per order (order_items)
- Payments per order (order_payments)
- Reviews per order (order_reviews)

This is descriptive (not pass/fail) and will guide star schema design.

In [ ]:
items_per_order = order_items.groupby("order_id").size()
payments_per_order = order_payments.groupby("order_id").size()
reviews_per_order = order_reviews.groupby("order_id").size()

items_per_order.describe(), payments_per_order.describe(), reviews_per_order.describe()

In [ ]:
multi_payment_orders = int((payments_per_order > 1).sum())
multi_review_orders = int((reviews_per_order > 1).sum())
multi_item_orders = int((items_per_order > 1).sum())

multi_item_orders, multi_payment_orders, multi_review_orders

## Phase 1 complete: structural validation

At this point we have:
- Loaded all datasets reproductibly and portably
- Validated key table grains
- Verify core FK relationships and quantified small gaps
- Enforced referential integrity with documented filters ('*_model' tables)
- Confirmed PK uniqueness (including composite PK for order_items)

Next phase: define a star schema (dimensions + facts) and implement transformations in SQL.

## Modeling grain definition

Based on structural validation:

- 'orders' is one row per order.
- 'order_items_model' is one row per (order_id, order_item_id).
- 'order_payments' is one row per (order_id, payment_sequential).
- 'order_reviews_model' may contain multiple reviews per order.

Primary analytical fact table will be defined at the item level (order_items_model), as it captures revenue and product-level granularity.


## Phase 2 - Star Schema Implementation

Primary analytical fact table:
FactOrderItems (one row per order item).

This table will:
- Enforce referential integrity (using *_model tables)
- Integrate order-level attributes
- Prepare structure for SQL materialization

In [ ]:
#Join order_items_model with orders to bring customer and order-level attributes

fact_order_items = (
    order_items_model
    .merge(
        orders[[
            "order_id",
            "customer_id",
            "order_status",
            "order_purchase_timestamp"
        ]],
        on="order_id",
        how="inner"
    )
)

fact_order_items.shape

## FactOrderItems (model-ready)

**Grain:** one row per ('order_id','order_item_id')

This fact integrates:
- Item-level measures: 'price', 'freight_value'
- Product reference: 'product_id'
- Customer reference (via orders): 'customer_id'
- Order attributes: 'order_status', 'order_purchase_timestamp'


Next: build dimensions (Customers, Products, Date) and add surrogate keys.

In [ ]:
# Grain uniqueness check
grain_unique =~fact_order_items.duplicated(subset=["order_id","order_item_id"]).any()

# Null checks for core keys
nulls = {
    "order_id_nulls": int(fact_order_items["order_id"].isna().sum()),
    "order_item_id_nulls": int(fact_order_items["order_item_id"].isna().sum()),
    "product_id_nulls": int(fact_order_items["product_id"].isna().sum()),
    "customer_id_nulls": int(fact_order_items["customer_id"].isna().sum()),
}

grain_unique, nulls

## Dimensions

We build three core dimensions:

- **DimCustomers** keyed by 'customer_id'
- **DimProducts** keyed by 'product_id'
- **DimDate** derived from 'order_purchase_timestamp' (purchase date)

Dimensions are de-duplicated and will later recieve surrogate keys in SQL.

In [ ]:
dim_customers = (
    customers[[
        "customer_id",
        "customer_unique_id",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state",
    ]]
    .drop_duplicates(subset=["customer_id"])
    .reset_index(drop=True)
)

dim_customers.shape

In [ ]:
dim_products = (
    products[[
        "product_id",
        "product_category_name",
        "product_category_name_english",
        "product_photos_qty",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm",
    ]]
    .drop_duplicates(subset=["product_id"])
    .reset_index(drop=True)
)

dim_products.shape

In [ ]:
# Ensure timestamp column is a datetime-like (required for .dt accessor)
fact_order_items["order_purchase_timestamp"] = pd.to_datetime(
    fact_order_items["order_purchase_timestamp"],
    errors="coerce"
)

#Sanity check
fact_order_items["order_purchase_timestamp"].dtype, int(fact_order_items["order_purchase_timestamp"].isna().sum())

In [ ]:
# Create a date dimension based on purchase date (not timestamp)
fact_order_items["purchase_date"] = fact_order_items["order_purchase_timestamp"].dt.normalize()

dim_date = (
    pd.DataFrame({"purchase_date": fact_order_items["purchase_date"].dropna().unique()})
    .assign(
        date_key=lambda d: d["purchase_date"].dt.strftime("%Y%m%d").astype(int),
        year=lambda d: d["purchase_date"].dt.year,
        month=lambda d: d["purchase_date"].dt.month,
        day=lambda d: d["purchase_date"].dt.day,
        weekday=lambda d: d["purchase_date"].dt.dayofweek,
    )
    .sort_values("purchase_date")
    .reset_index(drop=True)
)

dim_date.shape, dim_date.head()

## Star schema keys (logical)

We keep natural keys in the notebook:
- DimCustomers: 'customer_id'
- DimProducts: 'product_id'
- DimDate: 'date_key' derived from 'purchase_date'
In the warehouse (SQL layer), these can become surrogate keys if desired, but natural keys are sufficient for this model.

In [ ]:
# Add date_key to fact table (join on purchase_date)
fact_order_items = fact_order_items.merge(
    dim_date[["purchase_date","date_key"]],
    on="purchase_date",
    how="left"
)

#Sanity checks
int(fact_order_items["date_key"].isna().sum()), fact_order_items.shape

In [ ]:
# Dimension coverage checks (should be 10)% now, given previous filtering)
cust_missing = int((~fact_order_items["customer_id"].isin(dim_customers["customer_id"])).sum())
prod_missing = int((~fact_order_items["product_id"].isin(dim_products["product_id"])).sum())
date_missing = int(fact_order_items["date_key"].isna().sum())

cust_missing, prod_missing, date_missing

## End of Phase 2

## Beginning of Phase 3 — SQL materialization (DuckDB)

The star schema has been validated and assembled in pandas.
Next, we materialize the model in DuckDB using SQL scripts stored in `03-sql/`.

Execution order:
1. `03-sql/01_dim_date.sql`
2. `03-sql/02_dim_customers.sql`
3. `03-sql/03_dim_products.sql`
4. `03-sql/04_fact_orders.sql`
5. `03-sql/04_fact_order_items.sql`
6. `03-sql/04_fact_order_payments.sql`
7. `03-sql/04_fact_order_reviews.sql`